In this page, you will learn how to:

- run a `QuantumProgram` on local emulators
- choose between different emulator backends
- inspect and extract local execution results
- connect to Pasqal Cloud and run programs remotely
- submit remote jobs and retrieve their results
- execute compiled programs on a QPU

A `QuantumProgram` can be easily run on multiple backends provided by Pasqal:
- locally installed emulators
- remote cloud emulators
- QPUs

Remote emulators and QPU **require** credentials to submit a job.
More information on how to access a QPU through your favorite cloud provider, is available at [Pasqal's website](https://www.pasqal.com/solutions/cloud/).
Later, we will briefly show how to authenticate and send a remote job.

## A simple quantum program

Let us revisit the quantum program definition described before in the [Quantum programs](../quantum_program/index.md) page.

In [105]:
from qoolqit import Drive, Ramp, Register, Constant
from qoolqit import QuantumProgram
from qoolqit import MockDevice

# Create the register
register = Register.from_coordinates([(0,1), (0,-1), (2,0)])

# Defining the drive parameters
omega = 0.8
delta_i = -2.0 * omega
delta_f = -delta_i
T = 25.0

# Defining the drive
wf_amp = Constant(T, omega)
wf_det = Ramp(T, delta_i, delta_f)
drive = Drive(amplitude = wf_amp, detuning = wf_det)

# Creating the program
program = QuantumProgram(register, drive)

# Compiling the Program
device = MockDevice()
program.compile_to(device)

In the following sections we will provide more details on local/remote emulations and on running on available QPUs.

## Local Emulator

Local emulators run quantum simulations directly on your machine. They are ideal for development, testing, and small-scale quantum programs.
QoolQit provides easy access to local emulation through the `LocalEmulator` class, which allows you to run quantum programs locally.

In [106]:
from qoolqit.execution import LocalEmulator

emulator = LocalEmulator()
results = emulator.run(program)

### Backend Types

The `LocalEmulator` allows to emulate the program run on different backends provided by Pasqal:

- `QutipBackendV2`: Based on Qutip, runs programs with up to ~12 qubits and return qutip objects in the results (default).
- `SVBackend`: PyTorch based state vectors and sparse matrices emulator. Runs programs with up to ~25 qubits and return torch objects in the results. Requires installing the `emu-sv` package.
- `MPSBackend`: PyTorch based emulator using Matrix Product States (MPS). Runs programs with up to ~80 qubits and return torch objects in the results. Requires installing the `emu-mps` package.

To use a particular backend it is sufficient to specify it through the `backend_type` argument:

In [107]:
from qoolqit.execution import LocalEmulator, BackendType

emulator = LocalEmulator(backend_type=BackendType.QutipBackendV2)

### Handling local results

The call `emulator.run(program)` will return a `Sequence[Results]` object type. This is where the results of the computation are stored.
For more info about this specific object, please, have a look at [Pulser documentation](https://pulser.readthedocs.io/en/stable/apidoc/_autosummary/pulser.backend.Results.html#results).
As an example, lets inspect the results we got in the previous run:

In [108]:
# single result in the sequence
results[0].get_result_tags()

['bitstrings']

Then the bitstrings can be extracted simply as:

In [109]:
# single result in the sequence
final_bitstrings = results[0].final_bitstrings
final_bitstrings

Counter({'111': 800,
         '110': 72,
         '011': 59,
         '101': 54,
         '010': 8,
         '100': 5,
         '001': 2})

## Remote Emulator

Remote emulators run on cloud infrastructure, allowing you to simulate larger quantum systems without local computational constraints.
As anticipated, for remote workflows **credentials to create a connection are required**.
Here we will show how to create the specific handler of Pasqal Cloud services.
Again, for more information about Pasqal Cloud and other providers, please refer to the [Pasqal Cloud website](https://www.pasqal.com/solutions/cloud/).

Let's first initialize a connection. Without providing the actual credentials, we can only create an empty connection object, for displaying purposes only:

In [110]:
from pulser_pasqal import PasqalCloud

# connection = PasqalCloud(
#     username=USERNAME,  # Your username or email address for the Pasqal Cloud Platform
#     password=PASSWORD,  # The password for your Pasqal Cloud Platform account
#     project_id=PROJECT_ID,  # The ID of the project associated to your account
# )
connection = PasqalCloud()

To use such connection, and to send jobs to the cloud, we first need to initialize a remote emulator:

In [111]:
from qoolqit.execution import RemoteEmulator

emulator = RemoteEmulator(connection=connection)

As before, also `RemoteEmulator` can be instantiated with:
- `backend_type`: remote counterpart of local backends, namely `EmuFreeBackendV2` (default), `EmuSVBackend`, `EmuMPSBackend`.
- `emulation_config`: same as before — see [Emulation configuration](../../extended_usage/execution_config.md) for details.
- `runs`: same as before.

As an example, below, we specify to emulate the program with the `EmuMPSBackend` and a custom `EmulationConfig`:

In [112]:
from qoolqit.execution import (
    BackendType,
    EmulationConfig,
    Occupation,
    RemoteEmulator,
)

observables = (Occupation(evaluation_times=[0.5, 1.0]),)
emulation_config = EmulationConfig(observables=observables, with_modulation=True)

# initialize the remote emulator with the custom configuration and runs
remote_emulator = RemoteEmulator(
    backend_type=BackendType.EmuMPSBackend,
    connection=connection,
    emulation_config=emulation_config,
    runs=1000,
)

### Handling remote results

Remote emulators and QPU both have a `run()` method that will return a `Sequence[Results]` object type.
However, if your program requires intensive resources to be run, or if QPU happens to be on maintenance, the use of this method is discouraged since it might leave your script hanging.
In these situations prefer the use of the `submit(program) -> RemoteResults` instead:

```python
remote_emulator = RemoteEmulator(.., connection=connection, ...)
remote_results = remote_emulator.submit(program)
```

Here, the remote results can act as a job handler:
- Query the batch status: PENDING, RUNNING, DONE, etc.:
    ```python
    batch_status = remote_results.get_batch_status()
    ```
- Query the batch id, to be saved for later retrieval of results:
    ```python
    batch_id = remote_results.get_batch_id()
    ```
- Retrieve the remote results from `batch_id` and a `connection`:
    ```python
    from qoolqit.execution import RemoteResults

    remote_results = RemoteResults(batch_id, connection)
    ```

Once the batch has been completed (`batch_status` returns DONE), the complete results can be finally fetched as:
```python
results = remote_results.results
```

## Executing remotely on a QPU

A connection object can also be used to run the program directly on a QPU.

Then, to see the list of available devices, run:

In [113]:
connection.fetch_available_devices()

{'FRESNEL': FRESNEL}

A default QPU device is always available in the empty Pasqal Cloud connection, for demonstration purposes.

Now, to submit the program to the QPU, the program must be compiled for the specific QPU device, and then submitted to the QPU:

In [ ]:
from qoolqit.devices import Device

device = Device.from_connection(connection, "FRESNEL")
program.compile_to(device=device)

Finally, to set up a QPU backend, there is no configuration and, as per the properties of the quantum hardware, results will come as a bitstrings counter of length specified by the `runs` parameter.

In [115]:
from qoolqit.execution import QPU

qpu = QPU(connection=connection, runs=500)

Finally, submission and results handling work the same as for remote emulators.

## Hardware Considerations

When running on QPUs, consider:

- **Limited shots**: QPU time is precious, so choose your number of runs carefully
- **Hardware constraints**: Your program must be compiled for the specific QPU device
- **Queue times**: QPU jobs may wait in queue before execution

## Summary

This notebook covered the three main execution backends in QoolQit:

1. **Local Emulators**: Fast, local simulation for development and testing
2. **Remote Emulators**: Cloud-based simulation for larger quantum systems
3. **QPUs**: Real quantum hardware execution

Each backend has its use cases, and the choice depends on your specific needs:
- Use local emulators for quick prototyping and small systems
- Use remote emulators for larger simulations beyond local capabilities
- Use QPUs for real quantum experiments and final validation

For more detailed configuration options, see the [Extended Usage](../../extended_usage/execution_config.md) documentation.